# Mesh Curvature #

In this assignment, you will explore curvature on discrete surfaces. First you compute the Gaussian curvature and then the Mean curvature normal. Finally, you compute the shape operator for each vertex and use it to find both the mean curvature (again) and also the principal curvatures.

In [1]:
from pygel3d import hmesh, jupyter_display as jd
from numpy import zeros, dot
from numpy.linalg import norm
import numpy as np
from math import pi, acos, tan, acos, cos, sqrt
jd.set_export_mode(True)

def normalize(v):
    return v / np.linalg.norm(v)

def vertex_areas(m):
    pos = m.positions()
    face_areas = [ m.area(f) for f in m.faces() ]
    ora = zeros(len(pos))
    for v in m.vertices():
        edges = [normalize(pos[vn]-pos[v]) for vn in m.circulate_vertex(v, mode='v')]
        areas = [face_areas[f] for f in m.circulate_vertex(v, mode='f')]
        indices = range(len(edges))
        angles = [acos(dot(edges[i],edges[(i+1)%len(edges)])) for i in indices]
        ora[v] = sum([areas[i]*angles[i]/pi for i in indices])
    return ora

In [18]:
# A couple of meshes are provided in the zip. Feel free to try more.
#m = hmesh.obj_load("bunnygtest.obj")
m = hmesh.obj_load("Torus.obj")
hmesh.triangulate(m)
jd.display(m)

## Part 1: Compute and Integrate Gaussian Curvature

Compute the Gaussian curvature at all vertices. Visualize the Gaussian curvature on the mesh and compute the integral gaussian curvature over the mesh. Print out the result of the integral Gaussian curvature. Try this both for the bunny and the torus. Note that when visualizing the curvature, some vertices are likely to have extreme values. You probably need to clamp the values to a range in order to be able to get a meaningful visualization.

In [ ]:
K = zeros(len(m.vertices()))
pos = m.positions()
kint = 0.0
A = vertex_areas(m)

for v in m.vertices():
    # Insert code here which computes the Gaussian curvature for each vertex,
    # adds to the sum (kint), and assigns the value to K[v]
    
    # Your code here
    # ------------------
    angle_sum = 0.0

    for f in m.circulate_vertex(v, mode='f'):
        verts = list(m.circulate_face(f, mode='v'))

        i = verts.index(v)
        v1 = verts[(i + 1) % 3]
        v2 = verts[(i + 2) % 3]

        a = normalize(pos[v1] - pos[v])
        b = normalize(pos[v2] - pos[v])

        angle_sum += acos(np.clip(dot(a, b), -1.0, 1.0))

    defect = 2 * pi - angle_sum
    K[v] = defect / A[v]
    kint += defect       # equivalent to K[v] * A[v]

print("Integral Gaussian curvature:", kint)
jd.display(m, data=K)

Integral Gaussian curvature: -2.601474591301667e-12


## Part 2: Compute Mean Curvature Normal
In this exericse, you should compute the mean curvature normal $\mathbf{H} = \text{H}\cdot \mathbf{n}$ for each vertex in the mesh. On the mesh, we plot the length of this vector which is the mean curvature.

In [ ]:
def normalize(v):
    return v / np.linalg.norm(v)

def cotan_angle(m, i, j, k):
    # Compute and return the cotan

    # Your code here
    # ------------------
    
    pos = m.positions()
    a = pos[i] - pos[j]
    b = pos[k] - pos[j]
    return dot(a, b) / norm(np.cross(a, b))

def mean_curvature_normal(m):
    N = m.no_allocated_vertices()
    Hn = np.zeros((N,3))
    
    # Insert code here which computes the Mean curvature normal H*n for each vertex 
    # using the cotan formula.
    
    pos = m.positions()
    areas = vertex_areas(m)
    for v in m.vertices():
        
        # Your code here
        # ------------------
        weighted_sum = np.zeros(3)
        for f in m.circulate_vertex(v, mode='f'):
            verts = list(m.circulate_face(f, mode='v'))
            i = verts.index(v)
            v1 = verts[(i + 1) % 3]
            v2 = verts[(i + 2) % 3]

            cot_at_v2 = cotan_angle(m, v, v2, v1)
            cot_at_v1 = cotan_angle(m, v, v1, v2)
            weighted_sum += cot_at_v2 * (pos[v] - pos[v1])
            weighted_sum += cot_at_v1 * (pos[v] - pos[v2])

        Hn[v] = weighted_sum / (2 * areas[v])
        
    return Hn

In [21]:
# Display Mean Curvature
Hn = mean_curvature_normal(m)
H = norm(Hn,axis=1)
for v in m.vertices():
    H[v] *= np.sign(np.dot(Hn[v], m.vertex_normal(v)))
print("Mean Curvature", H.shape)
jd.display(m,data=H)

Mean Curvature (9216,)


## Part 3: Compute the Shape Operator
In this part, you  compute the shape operator $\mathbf{S} = - \begin{bmatrix} a & b \\ b & c \end{bmatrix}$ and use it to find the principal curvatures $\text{kmin}$ and $\text{kmax}$ and the mean curvature $\text{H}$.

In [ ]:
def shape_operator(m, v):
    pos = m.positions()

    # Insert code here which computes the shape operator matrix S that can be used to compute
    # the principal curvatures kmin and kmax and the mean curvature H
    
    # Your code here
    # ------------------
    
    # Principal curvature
    p = pos[v]
    n = normalize(m.vertex_normal(v))

    # Choose an orthonormal basis for the tangent plane at v.
    neighbors = list(m.circulate_vertex(v, mode='v'))
    tangent = pos[neighbors[0]] - p
    tangent -= dot(tangent, n) * n
    t1 = normalize(tangent)
    t2 = normalize(np.cross(n, t1))

    # Fit the local Monge patch z = (a*x^2 + 2*b*x*y + c*y^2)/2.
    A = []
    z = []
    for vn in neighbors:
        d = pos[vn] - p
        x = dot(d, t1)
        y = dot(d, t2)
        A.append([0.5 * x * x, x * y, 0.5 * y * y])
        z.append(dot(d, n))

    a, b, c = np.linalg.lstsq(np.asarray(A), np.asarray(z), rcond=None)[0]
    S = -np.array([[a, b], [b, c]])

    # The eigenvalues of S are the principal curvatures.
    kmin, kmax = np.linalg.eigvalsh(S)
    # Mean curvature
    H = 0.5 * (kmin + kmax)

    return kmin, kmax, H

In [7]:
# Display Mean Curvature
H = zeros(len(m.vertices()))
for v in m.vertices():
    kmin, kmax, mean_curvature = shape_operator(m,v)
    H[v] = mean_curvature
print("Mean Curvature")
jd.display(m,data=H)

Mean Curvature


In [8]:
# Display Principal Curvatures
Hn = mean_curvature_normal(m)
min_curvature = zeros(len(m.vertices()))
max_curvature = zeros(len(m.vertices()))
for v in m.vertices():
    kmin, kmax, mean_curvature = shape_operator(m,v)
    min_curvature[v] = kmin
    max_curvature[v] = kmax
print("Principal curvatures")
jd.display(m,data=min_curvature)
jd.display(m,data=max_curvature)

Principal curvatures


### Questions

- In the plot of Gaussian curvature, what differentiates regions with positive curvature from regions with negative curvature?
> Answer: The negative regions are quite pink while the positive regions are green between them are wide as it gradient of colors. The positive curvature regions are convex (curves outward), while the negative curvature regions are concave (curves inward).
- Would you expect the image to look different for mean curvature and, if so, how?
> Answer: Yes I would expect the image to look different. Mean curvature takes the average of the principal curvatures while Gaussian curvature takes the product of the principal curvatures. Principal curvatures are the maximum and minimum curvatures at a point on the surface. This means that the mean curvature image will look different because it represents the average of the two principal curvatures rather than their product. It may also have a smoother colour distribution with fewer extreme values, depending on the surface. 
- Explain the value you got for the integral Gaussian curvature. Why is it different for the bunny and the torus?
> Answer: Torus: -2.60e-12 \
Sphere: 12.566 \
Bunny: 12.566 \
The torus has almost an equal amount of positive and negative Gaussian curvature, which makes the integral almost zero as they cancel each other out. The bunny has an integral close to $4\pi$ which matches the Gauss-Bonnet theorem for an object that has the topology of a closed surface without holes.

- On the torus, why is the Gaussian curvature positive and negative, but the mean curvature always positive?
> Answer: The Gaussian curvature is the product of the two principal curvatures, while the mean curvature is the average of them. On the outer part of the torus both principal curvatures have the same sign, which results in a positive Gaussian curvature, while on the inner part they have opposite signs resulting in negative Gaussian curvature. The mean curvature remains positive because the positive principal curvature is larger than the negative one which makes the average positive. 
- On the torus, the maximum principal curvature is almost constant, but the minimum curvature varies a lot. Why is that?
> Answer: The maximum principal curvature is almost constant because the torus has a relatively uniform shape in one direction. However, the minimum principal curvature varies a lot because the torus has regions where it curves more sharply in the other direction. This variation is caused by the torus's circular shape, where the curvature changes depending on the position around the tube.